# Document Summariser Prototype
Extractive (TextRank) vs abstractive (BART, Pegasus) summarisation for news and policy articles.

Run top to bottom after placing `data/articles.jsonl` (100 articles, 30 with human reference summaries).

In [ ]:
!pip install -q -r requirements.txt

## 1. Load and clean the corpus

In [ ]:
import json, pandas as pd
from src.preprocess import clean_text

docs = [json.loads(l) for l in open('data/articles.jsonl', encoding='utf-8')]
for d in docs:
    d['clean'] = clean_text(d['text'])
    d['n_words_raw'] = len(d['text'].split())
    d['n_words_clean'] = len(d['clean'].split())
df = pd.DataFrame(docs)
print(len(df), 'articles;', (df.n_words_raw - df.n_words_clean).mean().round(1), 'words of boilerplate removed on average')
df[['n_words_raw','n_words_clean']].describe()

In [ ]:
refd = [d for d in docs if d.get('reference')]
cut = len(refd)//3
val, test = refd[:cut], refd[cut:]   # 10 validation / 20 test
print(len(val), len(test))

## 2. Extractive baseline (TextRank)

In [ ]:
from src.extractive import textrank_summary
from src.evaluate import rouge, flagged_rate, timed

ext_out = [timed(textrank_summary, d['clean']) for d in test]
ext_preds = [o[0] for o in ext_out]
refs = [d['reference'] for d in test]
srcs = [d['clean'] for d in test]
print('TextRank', rouge(ext_preds, refs), 'mean latency', sum(o[1] for o in ext_out)/len(ext_out))

## 3. Abstractive models with chunk-and-merge
Long articles are split into sentence-aligned chunks, each chunk is summarised, and the partial summaries are merged (and condensed again if still too long).

In [ ]:
from src.abstractive import Summariser
bart = Summariser('bart')
pegasus = Summariser('pegasus')
long_doc = max(docs, key=lambda d: d['n_words_clean'])
print(len(bart.chunk(long_doc['clean'])), 'chunks for the longest article')

## 4. Tune length, beam size and chunk overlap on the validation split

In [ ]:
from src.tune import grid_search, save
val_docs = [d['clean'] for d in val]; val_refs = [d['reference'] for d in val]
best_bart, all_bart = grid_search(bart, val_docs, val_refs)
best_peg, all_peg = grid_search(pegasus, val_docs, val_refs)
print(best_bart, best_peg)
save('results/tuning_bart.json', best_bart, all_bart); save('results/tuning_pegasus.json', best_peg, all_peg)

## 5. Evaluate on the test split: ROUGE-1/2/L, automatic hallucination screen, latency

In [ ]:
def run(model, best):
    out = [timed(model.summarise, d['clean'], max_len=best['max_len'], min_len=int(best['max_len']*.3), beams=best['beams'], overlap_sents=best['overlap'], use_cache=False) for d in test]
    return [o[0] for o in out], sum(o[1] for o in out)/len(out)

bart_preds, bart_lat = run(bart, best_bart)
peg_preds, peg_lat = run(pegasus, best_peg)

rows = {}
for name, preds, lat in [('TextRank', ext_preds, 0), ('BART', bart_preds, bart_lat), ('Pegasus', peg_preds, peg_lat)]:
    rows[name] = {**rouge(preds, refs), 'auto_flagged_%': flagged_rate(preds, srcs), 'latency_s': round(lat,2)}
pd.DataFrame(rows).T

## 6. Human rating (15 test articles)
Write the sheet, have raters fill faithfulness / coverage / fluency (1-5) and hallucination (y/n), then aggregate.

In [ ]:
from src.evaluate import write_rating_sheet, aggregate_ratings
write_rating_sheet('results/human_rating_sheet.csv', [d['id'] for d in test[:15]], {'TextRank': ext_preds[:15], 'BART': bart_preds[:15], 'Pegasus': peg_preds[:15]})
# after rating:
# aggregate_ratings('results/human_rating_sheet.csv')

## 7. Stretch: latency and memory on CPU, caching trade-off

In [ ]:
from src.evaluate import peak_memory_mb
import time
doc = test[0]['clean']
for name, m in [('BART', bart), ('Pegasus', pegasus)]:
    t=time.perf_counter(); m.summarise(doc, use_cache=False); cold=time.perf_counter()-t
    m.summarise(doc, use_cache=True)
    t=time.perf_counter(); m.summarise(doc, use_cache=True); hot=time.perf_counter()-t
    print(name, 'uncached', round(cold,2), 's | cached', round(hot,4), 's | peak mem', peak_memory_mb(), 'MB')

## 8. Success criteria check
- ROUGE-L beats extractive baseline by the agreed margin
- Hallucination below 10% of rated samples
- Summary returned within 10 seconds

Fill the final recommendation and limitations in the report.

In [ ]:
print('ROUGE-L gain BART over TextRank :', round(rows['BART']['rougeL']-rows['TextRank']['rougeL'],2))
print('ROUGE-L gain Pegasus over TextRank:', round(rows['Pegasus']['rougeL']-rows['TextRank']['rougeL'],2))
print('Within 10 s? BART', rows['BART']['latency_s']<=10, '| Pegasus', rows['Pegasus']['latency_s']<=10)